# 03. 뉴스 감성 분석
FinBERT로 뉴스 제목 감성 분석 후 종목·날짜별 집계 → `data/sentiment.csv`

> **GPU 런타임 권장**: 런타임 → 런타임 유형 변경 → T4 GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers

In [ ]:
# ── 설정 (본인 Drive 경로에 맞게 수정) ───────────────────────────────────────
DATA_DIR  = "/content/drive/MyDrive/term_project/data"
NEWS_PATH = "/content/drive/MyDrive/term_project/kospi_news.csv"

import os
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 디바이스: {device}")

In [ ]:
# ── KR-FinBERT 로드 ───────────────────────────────────────────────────────────
# ProsusAI/finbert (영어) → snunlp/KR-FinBert-SC (한국어 금융 특화)
MODEL_NAME = "snunlp/KR-FinBert-SC"
print(f"모델 로드 중: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

# 모델별 레이블 인덱스를 동적으로 파악 (모델마다 id2label 순서가 다름)
id2label  = model.config.id2label
label2idx = {v.lower(): k for k, v in id2label.items()}
POS_IDX = label2idx.get("positive", label2idx.get("긍정", 0))
NEG_IDX = label2idx.get("negative", label2idx.get("부정", 1))
NEU_IDX = label2idx.get("neutral",  label2idx.get("중립", 2))

print(f"로드 완료  |  레이블 매핑: {id2label}")
print(f"  positive idx={POS_IDX}, negative idx={NEG_IDX}, neutral idx={NEU_IDX}")


In [ ]:
# ── 감성 분석 함수 ────────────────────────────────────────────────────────────
def predict_sentiment(texts: list, batch_size=64) -> list:
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True,
                           max_length=128, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1).cpu()
        for p in probs:
            results.append({
                "positive": p[POS_IDX].item(),
                "negative": p[NEG_IDX].item(),
                "neutral":  p[NEU_IDX].item(),
            })
        if (i // batch_size) % 10 == 0:
            print(f"  {i + len(batch)}/{len(texts)}건 완료")
    return results


def add_sentiment_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["종목코드"] = df["종목코드"].astype(str).str.zfill(6)
    df["date"]    = pd.to_datetime(df["date"]).dt.normalize()
    df = df.sort_values(["종목코드", "date"]).reset_index(drop=True)

    for col in ["sentiment_mean", "sentiment_std", "news_count"]:
        df[col] = df[col].fillna(0)

    g = df.groupby("종목코드", group_keys=False)
    df["sentiment_lag1"]   = g["sentiment_mean"].shift(1)
    df["sentiment_lag2"]   = g["sentiment_mean"].shift(2)
    df["sentiment_change"] = g["sentiment_mean"].diff()

    rolling_mean = g["news_count"].transform(
        lambda x: x.shift(1).rolling(window=20, min_periods=5).mean()
    )
    rolling_std = g["news_count"].transform(
        lambda x: x.shift(1).rolling(window=20, min_periods=5).std()
    )
    df["news_count_zscore_20"] = (df["news_count"] - rolling_mean) / (rolling_std + 1e-6)

    new_cols = ["sentiment_lag1", "sentiment_lag2", "sentiment_change", "news_count_zscore_20"]
    df[new_cols] = df[new_cols].replace([np.inf, -np.inf], 0).fillna(0)
    return df


In [ ]:
# ── 뉴스 데이터 로드 ──────────────────────────────────────────────────────────
# kospi_news.csv 를 Drive에 업로드해야 합니다
df_news = pd.read_csv(NEWS_PATH)
df_news = df_news.dropna(subset=["제목"])
print(f"뉴스 {len(df_news)}건 로드")
df_news.head()

In [ ]:
# ── 감성 분석 수행 ────────────────────────────────────────────────────────────
texts = df_news["제목"].tolist()
print(f"감성 분석 수행 중... ({len(texts)}건)")
sentiments = predict_sentiment(texts)

df_news["sentiment_pos"]   = [s["positive"] for s in sentiments]
df_news["sentiment_neg"]   = [s["negative"] for s in sentiments]
df_news["sentiment_neu"]   = [s["neutral"]  for s in sentiments]
df_news["sentiment_score"] = df_news["sentiment_pos"] - df_news["sentiment_neg"]
print("감성 분석 완료")

In [ ]:
# ── 날짜 파싱 & 종목·날짜별 집계 ─────────────────────────────────────────────
df_news["date"] = pd.to_datetime(
    df_news["날짜"].str.strip().str[:10], format="%Y.%m.%d", errors="coerce"
)
df_news = df_news.dropna(subset=["date"])

# 종목코드 혼합 타입(정수 5930 vs 문자열 "005930") 문제 방지: groupby 전에 정규화
df_news["종목코드"] = df_news["종목코드"].astype(str).str.zfill(6)

daily_sentiment = df_news.groupby(["종목코드", "date"]).agg(
    sentiment_mean=("sentiment_score", "mean"),
    sentiment_std =("sentiment_score", "std"),
    news_count    =("sentiment_score", "count"),
).reset_index()

daily_sentiment["sentiment_std"] = daily_sentiment["sentiment_std"].fillna(0)

print(f"집계 완료: {len(daily_sentiment)}행")
daily_sentiment.head()

In [ ]:
# ── 시계열 파생 피처 추가 & 저장 ─────────────────────────────────────────────
daily_sentiment = add_sentiment_time_features(daily_sentiment)

output_path = f"{DATA_DIR}/sentiment.csv"
daily_sentiment.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"저장 완료: {output_path} ({len(daily_sentiment)}행)")
daily_sentiment.head()